<a href="https://colab.research.google.com/github/IanWorldHi/MLLearn/blob/main/pytorcher/torchWFFUNDCollab2Full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Full linear regression model

In [2]:
import torch
from torch import nn
import matplotlib.pyplot as plt
torch.__version__

'2.10.0+cu128'

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [5]:
weight = 0.7
bias = 0.3
start = 0
end = 1
step = 0.02

x = torch.arange(start, end, step).unsqueeze(dim=1)
y = weight * x + bias
x[:10], y[:10]

(tensor([[0.0000],
         [0.0200],
         [0.0400],
         [0.0600],
         [0.0800],
         [0.1000],
         [0.1200],
         [0.1400],
         [0.1600],
         [0.1800]]),
 tensor([[0.3000],
         [0.3140],
         [0.3280],
         [0.3420],
         [0.3560],
         [0.3700],
         [0.3840],
         [0.3980],
         [0.4120],
         [0.4260]]))

Using nn.Linear instead which is built into nn.Module, auto randoms it

In [6]:
class LinearRegressionModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layerer = nn.Linear(in_features=1, out_features=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layerer(x)
torch.manual_seed(42)
model_2 = LinearRegressionModel2()
model_2, model_2.state_dict()

(LinearRegressionModel2(
   (linear_layerer): Linear(in_features=1, out_features=1, bias=True)
 ),
 OrderedDict([('linear_layerer.weight', tensor([[0.7645]])),
              ('linear_layerer.bias', tensor([0.8300]))]))

In [7]:
model_2.to(device)
next(model_2.parameters()).device

device(type='cuda', index=0)

In [8]:
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.01)

We are putting the data on the device

In [10]:
epochs =1000
train_split = int(0.8 * len(x))

x_train = x[:train_split]
y_train = y[:train_split]
x_test = x[train_split:]
y_test = y[train_split:]
x_train = x_train.to(device)
y_train = y_train.to(device)
x_test = x_test.to(device)
y_test = y_test.to(device)

In [11]:
for epoch in range(epochs):
  model_2.train()
  y_pred = model_2(x_train)
  loss = loss_fn(y_pred, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  model_2.eval()
  with torch.inference_mode():
    test_pred = model_2(x_test)
    test_loss = loss_fn(test_pred, y_test)
  if epoch%100 == 0:
    print(f"Epoch: {epoch} | Loss: {loss} | Test Loss: {test_loss}")


Epoch: 0 | Loss: 0.5551779270172119 | Test Loss: 0.5739762187004089
Epoch: 100 | Loss: 0.006215683650225401 | Test Loss: 0.014086711220443249
Epoch: 200 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882
Epoch: 300 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882
Epoch: 400 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882
Epoch: 500 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882
Epoch: 600 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882
Epoch: 700 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882
Epoch: 800 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882
Epoch: 900 | Loss: 0.0012645035749301314 | Test Loss: 0.013801801018416882


In [12]:
model_2.eval()
with torch.inference_mode():
    y_preds = model_2(x_test)
y_preds

tensor([[0.8600],
        [0.8739],
        [0.8878],
        [0.9018],
        [0.9157],
        [0.9296],
        [0.9436],
        [0.9575],
        [0.9714],
        [0.9854]], device='cuda:0')

In [13]:
from pathlib import Path
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "01_pytorch_workflow_model_2.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME
torch.save(obj=model_2.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
loaded_model = LinearRegressionModel2()
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
loaded_model.to(device)
print(f"Loaded model:\n{loaded_model}")
print(f"Model on device:\n{next(loaded_model.parameters()).device}")
loaded_model.eval()
with torch.inference_mode():
    loaded_model_1_preds = loaded_model(x_test)
y_preds == loaded_model_1_preds